# 01 FedSTO DQA x MOE Trust Region 12h

FedSTOで増加した設定を本線にして、phase2の後半だけDQA x MOE residualをtraining loopへ入れる実験。DQA候補がsource/cloudyで悪化する場合はFedSTO-onlyを次roundへ渡す。

In [ ]:
from __future__ import annotations

import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

try:
    import pandas as pd
except Exception:
    pd = None

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "dynamic_quality_aware_classwise_aggregation").exists():
    REPO_ROOT = Path("/app/Object_Detection")

EXP_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation" / "dqa_moe_trust_region"
RUNNER = EXP_ROOT / "scripts" / "run_dqa_moe_trust_region.py"
WORKSPACE = EXP_ROOT / "output" / "01_fedsto_dqa_moe_trust_region_12h"
LOG_DIR = EXP_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RUNNER:", RUNNER, RUNNER.exists())
print("WORKSPACE:", WORKSPACE)


## Design

- FedSTOは変更しない: warmup=50, phase1=20, phase2=20。
- DQA x MOEはphase2 round 6からのみ。
- 各client checkpointをdomain expertとして扱い、自己生成されたvalidation metricでrouter weightを作る。
- `head`, `head+neck BN`, `neck+head` の複数候補を同じround内で試す。
- candidateがsource/cloudyで悪化したら採用しない。
- accepted candidateの中からproxy scoreが最大のものを次roundへ渡す。

In [ ]:
FULL_CMD = [
    sys.executable,
    str(RUNNER),
    "--workspace-root", str(WORKSPACE),
    "--warmup-epochs", "50",
    "--phase1-rounds", "20",
    "--phase2-rounds", "20",
    "--batch-size", "128",
    "--workers", "32",
    "--gpus", "2",
    "--master-port", "29541",
    "--dqa-start-round", "6",
    "--dqa-scope", "head_bn",
    "--dqa-search-candidates",
    "--dqa-candidate-scopes", "head,head_bn,neck_head",
    "--dqa-candidate-lambda-multipliers", "0.75,1.00",
    "--dqa-max-candidates", "6",
    "--dqa-lambda-start", "0.015",
    "--dqa-lambda-end", "0.05",
    "--dqa-max-relative-update", "0.01",
    "--dqa-acceptance-tolerance-map50", "0.003",
    "--dqa-acceptance-tolerance-map50-95", "0.002",
    "--val-batch-size", "32",
    "--run-final-eval",
]

print(" ".join(FULL_CMD))


In [ ]:
# Setup/dry-run check. This writes manifests/configs but does not train.
SETUP_CMD = [*FULL_CMD, "--setup-only", "--dry-run"]
print(" ".join(SETUP_CMD))
subprocess.run(SETUP_CMD, cwd=REPO_ROOT, check=True)


In [ ]:
# Full training guard.
# Set RUN_FULL = True to start the 12h-scale run.
RUN_FULL = False

if not RUN_FULL:
    print("RUN_FULL is False. Set it to True to launch training.")
else:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    log_path = LOG_DIR / f"01_fedsto_dqa_moe_trust_region_12h_{timestamp}.log"
    print("log:", log_path)
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.run(FULL_CMD, cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT)
    print("returncode:", proc.returncode)
    print(log_path.read_text(encoding="utf-8", errors="replace")[-12000:])
    if proc.returncode != 0:
        raise SystemExit(proc.returncode)


In [ ]:
summary_csv = WORKSPACE / "dqa_moe_round_summary.csv"
final_csv = WORKSPACE / "validation_reports" / "paper_protocol_eval_summary.csv"
print("round summary:", summary_csv, summary_csv.exists())
print("final eval:", final_csv, final_csv.exists())

if pd is not None and summary_csv.exists():
    display(pd.read_csv(summary_csv).tail(20))
elif summary_csv.exists():
    print(summary_csv.read_text(encoding="utf-8")[-4000:])

if pd is not None and final_csv.exists():
    display(pd.read_csv(final_csv))
elif final_csv.exists():
    print(final_csv.read_text(encoding="utf-8")[-4000:])
